<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Overture Maps Buildings in 3D

### CP101: Introduction to Urban Data Analytics

**Week 9, Wednesday** &nbsp;·&nbsp; Thematic Mapping - Map Types & Classification Schemes

**Estimated time:** 15-20 minutes &nbsp;·&nbsp; **Not graded, not submitted.**

<hr style="border: 1px solid #fdb515;" />

### How to work


- Run each code cell with `Shift + Enter`, in order, from the top.
- The map opens in a panel on the right of the notebook, and it is interactive.
  Drag to pan, scroll to zoom, and hold `Ctrl` while you drag to tilt and rotate.
  Hover over a building to see its height.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Learning Objectives

By the end of this notebook, you will be able to:

1. Fetch Overture Maps building footprints for a bounding box with the `overturemaps` library, using the latest data release
2. Rescale skewed building heights on a log scale with `LogNorm`
3. Draw a city's buildings in 3D at their real heights with Lonboard's `PolygonLayer`
4. Save an interactive map as a standalone `.html` file with `.to_html()`

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Overture Maps

[Overture Maps](https://overturemaps.org) is an open map dataset run by a foundation that Amazon, Meta, Microsoft and TomTom started.
It combines data from its members, civic organizations and open sources such as OpenStreetMap, and checks it for errors and vandalism before every release.

Its buildings layer alone has more than 2 billion footprints, so you never download all of it.
The [`overturemaps`](https://github.com/OvertureMaps/overturemaps-py) Python library asks for one bounding box, and only the buildings inside that box come back.
That works because Overture stores its data in the cloud as [GeoParquet](https://geoparquet.org/) files, which can be read in pieces, with no server in the middle.

**Credit.** This notebook is adapted from the [Overture Maps Buildings tutorial](https://developmentseed.org/lonboard/latest/examples/overture-maps/) on the Lonboard website.
[Lonboard](https://github.com/developmentseed/lonboard) is built by [Development Seed](https://developmentseed.org/) and released under the MIT License.

## Let's add a new feature to view our maps

[`Sidecar`](https://github.com/jupyter-widgets/jupyterlab-sidecar) opens a panel to the right of the notebook in JupyterLab, so the map stays in view while you scroll and run the cells below.
The next two cells install and import it, and `display()` is what puts the map inside that panel.

In [ ]:
!pip install sidecar -q

In [ ]:
from sidecar import Sidecar
from IPython.display import display
import ipywidgets as widgets

In [ ]:
from pathlib import Path

import geopandas as gpd
import lonboard
import overturemaps
import requests
from lonboard import Map, PolygonLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from matplotlib.colors import LogNorm
from palettable.colorbrewer.sequential import Oranges_9_r

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Data for this notebook

**What does Manhattan look like with every building drawn at its real height?**

A bounding box is four numbers, the west, south, east and north edges of an area in longitude and latitude.
We used [this bounding box tool](https://boundingbox.klokantech.com/) to draw a box around Manhattan, chose "CSV" in the drop-down menu and pasted the result into the next cell.
Four numbers separated by commas make a **tuple**.

In [ ]:
# west, south, east and north edges of Manhattan
bbox = -74.028907, 40.693052, -73.907955, 40.877889
type(bbox)

Overture publishes a new release of its data every month.
Its [STAC catalog](https://stac.overturemaps.org/catalog.json) is a small file that lists the releases, so we ask it for the latest one.

In [ ]:
# ask Overture's catalog for its newest release
response = requests.get("https://stac.overturemaps.org/catalog.json")
latest_release = response.json().get("latest")
print(f"The latest release is: {latest_release}")

`overturemaps.core.geodataframe()` takes the kind of feature we want (`"building"`), a bounding box and a release, and returns a GeoDataFrame.
It reads only the parts of Overture's files that fall inside the box.

All of Manhattan is about 25 MB of buildings, too much for the whole class to download at once.
So the next cell first looks for a saved copy at `data/raw/nyc_buildings.parquet`.
That copy is already in your folder, so the cell loads it in a few seconds and downloads nothing.
Without the saved copy, the cell would fetch the buildings inside `bbox` and save them, so the download happens only once.

In [ ]:
local_path = Path("data/raw/nyc_buildings.parquet")

if local_path.exists():
    nyc_gdf = gpd.read_parquet(local_path)
else:
    # download the buildings inside bbox once, then save them
    nyc_gdf = overturemaps.core.geodataframe(
        "building",
        bbox=bbox,
        release=latest_release
        )
    nyc_gdf.to_parquet(local_path)

# the saved file has no CRS, so set it to longitude and latitude (EPSG:4326)
if nyc_gdf.crs is None:
    nyc_gdf = nyc_gdf.set_crs(4326)

len(nyc_gdf)

In [ ]:
nyc_gdf.columns

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Coloring buildings by height

Start with the heights as a NumPy array.
About 5% of the buildings have no recorded height, so fill those with 1 meter, the bottom of our scale.

In [ ]:
# heights in meters, with missing heights set to 1
heights = (
    nyc_gdf["height"]
    .fillna(1)
    .to_numpy()
    )

Most buildings in Manhattan are low, and a few are very tall.
The median building in this file is about 10 meters tall, and the tallest is 472 meters.
On a straight (linear) color scale, almost every building would get the same dark color, and only a handful of towers would stand out.

[`LogNorm`](https://matplotlib.org/stable/api/_as_gen/matplotlib.colors.LogNorm.html) rescales the heights to values between 0 and 1 on a **log scale** instead.
On a log scale, every tenfold jump in height takes up the same share of the colors, so the gap between 1 and 10 meters is as wide as the gap between 10 and 100 meters.
`clip=True` keeps every value between 0 and 1.
In the Speedtest notebook a straight line between two bounds was enough, but heights are too skewed for that.

In [ ]:
normalizer = LogNorm(
    1,
    heights.max(),
    clip=True
    )
normalized_heights = normalizer(heights)

Next, pick a colormap.
On a dark basemap the brightest colors stand out most, so we use ColorBrewer's [`Oranges_9`](https://jiffyclub.github.io/palettable/colorbrewer/sequential/#oranges_9) reversed (`_r`).
The lowest buildings are dark brown, mid-height ones are orange, and the tallest are nearly white.

In [ ]:
Oranges_9_r.mpl_colormap

[`apply_continuous_cmap()`](https://developmentseed.org/lonboard/latest/api/colormap/#lonboard.colormap.apply_continuous_cmap) looks up each rescaled height on the colormap and returns one color per building, as red, green and blue values between 0 and 255.

In [ ]:
colors = (
    lonboard
    .colormap
    .apply_continuous_cmap(
        normalized_heights,
        Oranges_9_r
        )
)
colors

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Drawing Manhattan in 3D

A `PolygonLayer` draws each footprint.
`extruded=True` raises each footprint into a block, `get_elevation` sets the block's height in meters, and `get_fill_color` sets its color.
Keeping only the `height` and `geometry` columns means less data to send to your browser, so the map loads faster.

In [ ]:
layer = PolygonLayer.from_geopandas(
    nyc_gdf[["height", "geometry"]],
    extruded=True,
    get_elevation=heights,
    get_fill_color=colors
    )

The layer holds 144,790 buildings with about 1.27 million corner points (vertices), and your graphics card draws all of them.

`view_state` sets where the map opens.
`pitch` tilts the camera, where 0 looks straight down, and `bearing` turns it.
The basemap is CARTO's dark style, because 3D blocks keep their shape better against a dark background.
`show_tooltip=True` shows a building's height when you hover over it.

If you close the side panel by mistake, run the map cell again.
It closes any old panel first, so you only ever have one.

In [ ]:
view_state = {
    "longitude": -73.984,
    "latitude": 40.727,
    "zoom": 12.7,
    "pitch": 60,
    "bearing": 13
    }

m = Map(
    layer,
    view_state=view_state,
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatterNoLabels),
    show_tooltip=True
    )

m.height = "100vh"
# close the panel from an earlier run, so you keep only one
if "sc" in globals():
    sc.close()

# Display it beside the notebook.
sc = Sidecar(title="Spatial Analysis", anchor="split-right")

with sc:
    display(m)

### Save and share the map

`.to_html()` saves the map, data included, as one `.html` file next to this notebook.
To keep it, find `nyc_buildings_3d.html` in the file browser on the left, right-click it and choose **Download**.
Then open the downloaded file in any web browser, with no Python needed.
The file is about 22 MB, so the buildings appear a little after the basemap.

In [ ]:
# save the 3D map as a standalone HTML file
m.to_html(
    "nyc_buildings_3d.html",
    title="Manhattan Building Heights"
    )

### ✏️ Your Turn: Berkeley in 3D

Now draw a place you know.
`data/raw/berkeley_buildings.parquet` holds the 1,905 buildings of downtown Berkeley and the UC Berkeley campus.
We fetched them from Overture with the same call as Manhattan and kept only each building's `name`, `height` and footprint, so the file is small and loads in a moment.
Draw them in 3D and save the map as `berkeley_buildings_3d.html`.
Then download the file, open it and find the Campanile, the tallest building on the map at about 94 meters.

Work through the same steps as above, with new names such as `berkeley_gdf`, `berkeley_heights` and `berkeley_map`.

1. Load the buildings with `gpd.read_parquet()`.
2. Fill the missing heights, rescale them with `LogNorm` and turn them into colors.
3. Build a `PolygonLayer` and a `Map` whose `view_state` is centered on Berkeley.
4. Save the map with `.to_html()`.

<details>
<summary>Hint</summary>

- The file already has its CRS, so you do not need to set it.
- The center of the area is at about longitude `-122.2615` and latitude `37.8705`, and a `zoom` of 15 fits all of it.
- Keep the `name` column in the layer and pass `show_tooltip=True` to `Map()`, so hovering over a building shows its name.
- End the cell with `berkeley_map` to show the map right below it.
- About a third of the buildings have no recorded height, so they stay flat at 1 meter.

</details>

In [ ]:
# ✏️ Your Turn: downtown Berkeley and the UC Berkeley campus in 3D

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✅ Submit your Berkeley map in class (1 point)

Your Berkeley map is worth **1 point** if you submit it during class, and that point can cover one absence.

In class, before **2:30 pm**:

1. Run your Your Turn cell, including the `.to_html()` line, so `berkeley_buildings_3d.html` is saved next to this notebook.
2. Run the cell below.
   It checks that the file is there and prints its size.
3. In the file browser, right-click `berkeley_buildings_3d.html` and choose **Download**.
4. Open the downloaded file in your browser and check that the buildings appear and you can find the Campanile.
5. Upload `berkeley_buildings_3d.html` to the Gradescope assignment [ClassActivity_Lonboard_BerkeleyMap](https://www.gradescope.com/courses/1333040/assignments/8913315).

In [ ]:
# check that your Berkeley map was saved
berkeley_file = Path("berkeley_buildings_3d.html")
assert berkeley_file.exists(), "Run your Your Turn cell, including the .to_html() line, then run this again."
print(berkeley_file, "is saved,", round(berkeley_file.stat().st_size / 1_000_000, 1), "MB")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary - Overture Maps Buildings in 3D

| Step | What it does |
|---|---|
| STAC catalog query (`stac.overturemaps.org`) | Finds the latest Overture data release |
| `overturemaps.core.geodataframe("building", bbox=..., release=...)` | Fetches the building footprints inside a bounding box straight out of Overture's cloud files |
| `LogNorm` | Rescales skewed heights on a log scale, so a few towers do not wash out the low buildings |
| `lonboard.colormap.apply_continuous_cmap()` | Turns each rescaled height into a color on `Oranges_9_r` |
| `PolygonLayer` + `Map()` | Draws every footprint as a block at its real height, seen through a tilted `view_state` |
| `.to_html()` | Saves the interactive map as one shareable `.html` file |

## Key Takeaways

1. **Query, do not download** - a bounding box keeps only Manhattan's 144,790 of Overture's billions of buildings.
2. **Cloud GeoParquet needs no server** - `overturemaps` reads only the parts of the files inside the box.
3. **Skewed data needs a log scale** - with a median height of about 10 meters and a maximum of 472, a linear scale would give almost every building the same color.
4. **One variable can use two channels** - each block's height and its color both show how tall it is, so the low-rise blocks stay readable next to the towers.
5. **Lonboard draws 3D on your graphics card** - `extruded=True`, `get_elevation` and a tilted `view_state` turn flat footprints into a city.

## Further reading

- [Overture Maps documentation](https://docs.overturemaps.org/)
- [`overturemaps` Python library](https://github.com/OvertureMaps/overturemaps-py)
- [Lonboard documentation](https://developmentseed.org/lonboard/latest/)
- A [hosted version of Lonboard's original Overture example](https://notebooksharing.space/view/f16f8f4c7133c4576938b262f17ce617cf112106bc25a24a08b528f8f0d11828#displayOptions=) (26 MB download)